# Preprocessing

In [ ]:
!pip install neologdn mojimoji
!pip install sudachipy sudachidict_core
 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 266.6/266.6 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.2/211.2 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 MB 9.6 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.4/520.4 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.3/115.3 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.4/166.4 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.4/135.4 kB 10.2 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully

## **JParaCrawl**

<i>Because it too old so 2 first cell is turn of legacy to download</i>

In [ ]:
%%bash
cat > openssl_legacy.cnf <<EOF
openssl_conf = openssl_init

[openssl_init]
ssl_conf = ssl_sect

[ssl_sect]
system_default = system_default_sect

[system_default_sect]
Options = UnsafeLegacyRenegotiation
EOF

In [ ]:
import os
os.environ["OPENSSL_CONF"] = "/content/openssl_legacy.cnf"

In [ ]:
!wget --no-check-certificate \
https://www.kecl.ntt.co.jp/icl/lirg/jparacrawl/release/3.0/pretrained_models/ja-en/big.tar.gz

--2026-05-13 17:54:52--  https://www.kecl.ntt.co.jp/icl/lirg/jparacrawl/release/3.0/pretrained_models/ja-en/big.tar.gz
Resolving www.kecl.ntt.co.jp (www.kecl.ntt.co.jp)... 163.137.218.162
Connecting to www.kecl.ntt.co.jp (www.kecl.ntt.co.jp)|163.137.218.162|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2785877612 (2.6G) [application/x-gzip]
Saving to: ‘big.tar.gz’

big.tar.gz            0%[                    ]   4.14M   736KB/s    eta 67m 5s ^C


In [ ]:
!tar -xzf big.tar.gz


gzip: stdin: unexpected end of file
tar: Unexpected EOF in archive
tar: Unexpected EOF in archive
tar: Error is not recoverable: exiting now


## **KFTT**

In [2]:
from datasets import load_dataset

ds = load_dataset("may-ohta/kftt")
print(ds)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Generating tune split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['translation'],
        num_rows: 440289
    })
    validation: Dataset({
        features: ['translation'],
        num_rows: 1167
    })
    test: Dataset({
        features: ['translation'],
        num_rows: 1161
    })
    tune: Dataset({
        features: ['translation'],
        num_rows: 1236
    })
})


## Preprocessing

In [3]:
import spacy
from sudachipy import dictionary, tokenizer


### Super cleaner

In [ ]:
import re

import neologdn
from mojimoji import han_to_zen

ESCAPE_CODES = [r'&lt;', r'&gt;', r'&amp;', r'&quot;', r'&nbsp;', r'&copy;']

HIRAGANA = r'\u3041-\u3096'
KATAKANA = r'\u30A1-\u30F6'
PROLONGED_SOUND_MARK = r'\u30FC'
KANJI = r'\u3006\u4E00-\u9FFF'  # U+3006: 〆
REPEATING_MARK = r'\u3005'

WHITELIST_PTN = re.compile(rf'[a-zA-Z0-9!?()「」、。{HIRAGANA}{KATAKANA}{PROLONGED_SOUND_MARK}{KANJI}{REPEATING_MARK}]')
JP_PTN = re.compile(rf'[{HIRAGANA}{KATAKANA}{PROLONGED_SOUND_MARK}{KANJI}]')


def clean_text(text: str, twitter: bool, han2zen: bool, repeat: int = 3) -> str:
    text = _normalize(text=text, repeat=repeat)
    if _is_japanese(text):
        if twitter is True:
            text = _twitter_preprocess(text=text)
        text = _filter(text=text)
    if han2zen is True:
        text = han_to_zen(text)
    return text


def _normalize(text: str, repeat: int) -> str:
    return neologdn.normalize(text, repeat=repeat)


def _is_japanese(string: str) -> bool:
    al_num = re.compile(r'^[a-zA-Z0-9()!?,.:;\-\'\"\s]+$')
    return al_num.match(string) is None


def _twitter_preprocess(text: str) -> str:
    replaced_text = re.sub(r'[RT]\w+', '', text)
    replaced_text = re.sub(r'[@][a-zA-Z0-9_]+', '', replaced_text)
    replaced_text = re.sub(r'#(\w+)', '', replaced_text)
    return replaced_text


def _replace_punctuation(text: str) -> str:
    replaced_text = re.sub(r'、+', '、', text)  # "、、、" -> "、"
    replaced_text = re.sub(r'[、。]*。[、。]*', '。', replaced_text)
    replaced_text = re.sub(r'^[、。!?]', '', replaced_text)
    replaced_text = re.sub(
        rf'。[a-zA-Z0-9!?「」{HIRAGANA}{KATAKANA}{PROLONGED_SOUND_MARK}{KANJI}]。', '。', replaced_text)
    return replaced_text


def _whitelist_filter(text: str) -> str:
    """
    あいうw → あいう。
    (あいう)w → (あいう)。
    あいう → あいう。
    あいう☆ → あいう。
    あいう。w 。→ あいう。。。

    """
    ptn = re.compile(rf'[0-9。w{HIRAGANA}{KATAKANA}{PROLONGED_SOUND_MARK}{KANJI}]')
    filtered_text = ''
    for i, character in enumerate(text):
        if WHITELIST_PTN.match(character) and \
                not (character == 'w' and filtered_text and ptn.match(filtered_text[-1])):
            filtered_text += character
            continue
        filtered_text += '。'
    filtered_text += '。'
    return filtered_text


def _delete_kaomoji(text: str) -> str:
    text_ = ''
    buff = ''
    bracket_counter = 0
    for c in text:
        buff += c
        if c == '(':
            bracket_counter += 1
        elif c == ')':
            bracket_counter -= 1
            if bracket_counter == 0:
                stripped_buff = buff.lstrip('(').rstrip(')')
                if all(JP_PTN.match(c) for c in stripped_buff) and stripped_buff:
                    text_ += buff
                buff = ''
                continue
        if bracket_counter == 0:
            text_ += buff
            buff = ''
    return text_


def _filter(text: str) -> str:
    text = re.sub(r'(http|https)://([-\w]+\.)+[-\w]+(/[-\w./?%&=]*)?', '', text)
    for escape_code in ESCAPE_CODES:
        text = re.sub(escape_code, '', text)
    text = _whitelist_filter(text=text)
    text = _replace_punctuation(text)

    text = re.sub(r'笑笑+', '笑', text)
    text = re.sub(r'笑。', '。', text)

    text = re.sub(r'([!?。])[a-zA-Z0-9]+([!?。])', r'\1\2', text)
    text = _replace_punctuation(text)

    text = _delete_kaomoji(text)
    text = _replace_punctuation(text)

    text = re.sub(r'(。\))|(\(。)', '。', text)
    text = re.sub(r'[。!?][ノシﾉｼ]+[。!?]', '。', text)
    text = re.sub(r'。([!?])', r'\1', text)
    text = re.sub(r'([!?])。', r'\1', text)
    text = _replace_punctuation(text)

    text = re.sub(r'!!+', '!', text)
    text = re.sub(r'\?\?+', '?', text)
    text = re.sub(r'^.。', '', text)
    text = '' if len(text) == 1 else text

    return text

In [ ]:
JP_ONLY = re.compile(r'[\u3041-\u3096\u30A1-\u30F6\u30FC\u4E00-\u9FFF]+')

def filter(text: str) -> str:
    text = neologdn.normalize(text)
    text = han_to_zen(text)
    return ''.join([c for c in text if JP_ONLY.match(c)])
with open(INPUT, encoding = 'utf-8') as f, open(OUTPUT, 'w', encoding = 'utf-8') as out:
  for line in f:
    parts = line.strip().split()
    token, freq = parts
    if int(freq) >= MIN_FEQ:
      cleaned = filter(token).strip()
      if not cleaned:
        continue

      out.write(cleaned + '\n')



NameError: name 'INPUT' is not defined

### KFTT cleaner

In [8]:
import sentencepiece as spm
from tqdm import tqdm
import os

SAVE_DIR = "./kftt_processed"
os.makedirs(SAVE_DIR, exist_ok=True)

tokenizer_obj = dictionary.Dictionary().create()
mode = tokenizer.Tokenizer.SplitMode.B

def sudachi_tokenize(text):
    tokens = tokenizer_obj.tokenize(text, mode)
    return " ".join([tok.surface() for tok in tokens])

def save_split(split_name):

    ja_path = os.path.join(SAVE_DIR, f"{split_name}.ja")
    en_path = os.path.join(SAVE_DIR, f"{split_name}.en")

    with open(ja_path, "w", encoding="utf-8") as fja, \
         open(en_path, "w", encoding="utf-8") as fen:

        for item in tqdm(ds[split_name], desc=f"Processing {split_name}"):

            ja = item["translation"]["ja"].strip()
            en = item["translation"]["en"].strip()

            ja_tok = sudachi_tokenize(ja)
            en_tok = en.lower()

            fja.write(ja_tok + "\n")
            fen.write(en_tok + "\n")




In [9]:
sample = ds["train"][0]

ja_text = sample["translation"]["ja"]
en_text = sample["translation"]["en"]

print("RAW JA:")
print(ja_text)

print("\nTOKENIZED JA:")
print(sudachi_tokenize(ja_text))

RAW JA:
雪舟（せっしゅう、1420年（応永27年）-1506年（永正3年））は号で、15世紀後半室町時代に活躍した水墨画家・禅僧で、画聖とも称えられる。

TOKENIZED JA:
雪舟 （ せっ しゅう 、 1420 年 （ 応永 27 年 ） - 1506 年 （ 永正 3 年 ） ） は 号 で 、 15 世紀 後半 室町 時代 に 活躍 し た 水墨 画家 ・ 禅僧 で 、 画聖 と も 称え られる 。


In [10]:
save_split("train")
save_split("validation")
save_split("test")

print("\nSaved tokenized corpus!")

Processing test: 100%|██████████| 1161/1161 [00:00<00:00, 6624.94it/s]


Saved tokenized corpus!


# SentencePiece

In [30]:
!git clone https://github.com/google/sentencepiece.git

fatal: destination path 'sentencepiece' already exists and is not an empty directory.


In [31]:
cd sentencepiece

/content/sentencepiece


In [32]:
!apt-get update
!apt-get install -y cmake build-essential pkg-config libgoogle-perftools-dev

Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.2 MB]
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 10.2 MB in 2s (6,223 kB/s)
^C
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
build-essential is already the newest version (12.9ubuntu3).
libgoogle-perftools-dev is already the newest version (2.9.1-0ubuntu3).
pkg-config is already the newest version (0.

In [33]:
mkdir build

mkdir: cannot create directory ‘build’: File exists


In [34]:
cd build

/content/sentencepiece/build


In [35]:
!cmake ..

-- VERSION: 0.2.2
-- Populating abseil-cpp
-- Configuring done (0.0s)
-- Generating done (0.0s)
-- Build files have been written to: /content/sentencepiece/build/abseil-cpp-subbuild
[ 11%] Performing update step for 'abseil-cpp-populate'
-- Already at requested tag: 20260107.1
[ 22%] No patch step for 'abseil-cpp-populate'
[ 33%] No configure step for 'abseil-cpp-populate'
[ 44%] No build step for 'abseil-cpp-populate'
[ 55%] No install step for 'abseil-cpp-populate'
[ 66%] No test step for 'abseil-cpp-populate'
[ 77%] Completed 'abseil-cpp-populate'
[100%] Built target abseil-cpp-populate
-- Found TCMalloc: /usr/lib/x86_64-linux-gnu/libtcmalloc_minimal.so
-- Configuring done (0.5s)
-- Generating done (0.7s)
-- Build files have been written to: /content/sentencepiece/build


In [36]:
!make -j $(nproc)
!make install
!ldconfig

[  0%] Built target absl_spinlock_wait
[  1%] Built target absl_log_severity
[  2%] Built target absl_strerror
[  2%] Built target absl_utf8_for_code_point
[  5%] Built target absl_time_zone
[  5%] Built target absl_civil_time
[  6%] Built target absl_exponential_biased
[  6%] Built target absl_leak_check
[  7%] Built target absl_flags_commandlineflag_internal
[  8%] Built target absl_log_internal_nullguard
[  8%] Built target absl_periodic_sampler
[  9%] Built target absl_random_internal_platform
[ 12%] Built target sentencepiece_train-static
[ 13%] Built target absl_raw_logging_internal
[ 15%] Built target absl_random_internal_randen_hwaes_impl
[ 26%] Built target sentencepiece-static
[ 26%] Built target absl_random_internal_randen_hwaes
[ 26%] Built target absl_random_internal_randen_slow
[ 27%] Built target absl_throw_delegate
[ 28%] Built target absl_scoped_set_env
[ 30%] Built target absl_debugging_internal
[ 30%] Built target absl_cordz_functions
[ 31%] Built target absl_base
[ 

In [37]:
cd ../../

/content


In [38]:
!cat /content/kftt_processed/test.ja /content/kftt_processed/test.ja > combined_test.txt
!cat /content/kftt_processed/train.ja /content/kftt_processed/train.ja > combined_train.txt
!cat /content/kftt_processed/validation.ja /content/kftt_processed/validation.ja > combined_validation.txt

In [39]:
!spm_train --input=combined_train.txt --model_prefix=kftt_spm --vocab_size=16000 \
          --character_coverage=0.9995 --model_type=unigram \
          --pad_id=0 --unk_id=1 --bos_id=2 --eos_id=3

I0515 17:14:56.308937   18918 sentencepiece_trainer.cc:78] Starts training with : 
trainer_spec {
  input: combined_train.txt
  input_format: 
  model_prefix: kftt_spm
  model_type: UNIGRAM
  vocab_size: 16000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 1
  bos_id: 2
  eos_id: 3
  pad_id: 0
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surface:  ⁇ 
  enable_diffe

In [40]:
sp = spm.SentencePieceProcessor()
sp.load("kftt_spm.model")

def encode_file(input_path, output_path):

    with open(input_path, encoding="utf-8") as fin, \
         open(output_path, "w", encoding="utf-8") as fout:

        for line in fin:

            pieces = sp.encode(line.strip(), out_type=str)

            fout.write(" ".join(pieces) + "\n")

encode_file(
    "combined_train.txt",
    "train.spm"
)

encode_file(
    "combined_validation.txt",
    "validation.spm"
)

encode_file(
    "combined_test.txt",
    "test.spm"
)

In [41]:
def encode_sentence(ja_text, en_text):

    ja_tok = sudachi_tokenize(ja_text)

    ja_ids = sp.encode(ja_tok)

    en_ids = sp.encode(en_text.lower())

    return {
        "ja_tokens": ja_tok,
        "input_ids": ja_ids,
        "labels": en_ids
    }


res = encode_sentence("最小検出歪みは１０−５であった。",
        "As a result, it was found that its minimum detection strain was 10-5.")

print(res)

{'ja_tokens': '最小 検出 歪み は １０ − ５ で あっ た 。', 'input_ids': [1285, 5382, 10618, 8, 15110, 173, 9, 82, 8, 1, 72, 17, 45, 11, 7], 'labels': [8, 12254, 8, 1100, 8, 3376, 7444, 1742, 2802, 1822, 1775, 8, 8482, 8, 5356, 12254, 8, 6732, 1504, 1742, 3610, 2901, 8, 1822, 5188, 1822, 8, 8482, 2177, 706, 6242, 1209, 2309, 1742, 2309, 8, 12423, 13420, 3936, 13064, 5317, 8, 10560, 7365, 6242, 8, 5356, 12254, 82, 11899, 2422, 611]}


# NLLB

In [45]:
!pip install -U transformers huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.6/663.6 kB 36.4 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 2.14.7 requires huggingface-hub<1.0.0,>=0.14.0, but you have huggingface-hub 1.15.0 which is incompatible.


In [2]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("facebook/nllb-200-distilled-600M")
model = AutoModelForSeq2SeqLM.from_pretrained("facebook/nllb-200-distilled-600M")

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]